# Exp 12: does training on individual leaves help on field photos?

**Question.** The shipped model (v3) is right on 63.7% of PlantDoc field photos. It saw each PlantDoc training photo
as one example, even when the photo shows several leaves, and only 285 such photos exist. PlantDoc's object-detection
release has a box around every leaf (Singh et al. 2020). Does adding each boxed leaf as its own training example help?

**What changes (and nothing else).**
1. **Training:** the 285 whole photos stay, plus one cut-out per boxed leaf in those same photos (~1,000 leaves).
   Boxes come only from training photos; no validation or test photo is cut up.
2. **Prediction:** each photo's answer is averaged with its mirror image's. Reported with and without, so each
   change's effect is visible on its own.

Model, data mix, balancing, augmentation, optimiser, 10 epochs and the 1,306 test photos are all the same as v3.

**How to run.** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. About 35–45 minutes
(less if the photos are already on your Drive from `colab_setup.ipynb`). Keep this tab open.

## 1. GPU, Google Drive and the code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
REPO = "https://github.com/krishna-chhabra-cse/crop-doctor.git"
CODE = "/content/crop-doctor"
if os.path.exists(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
%cd {CODE}
!pip install -q timm

## 2. Photos: restore from Drive, or download once

In [ ]:
from pathlib import Path
LOCAL = Path("/content/datasets")
ARCHIVE = Path("/content/drive/MyDrive/crop_doctor/datasets.tar")
os.environ["CROP_DOCTOR_DATASETS"] = str(LOCAL)
if LOCAL.exists() and any(LOCAL.iterdir()):
    print("Photos already on this machine.")
elif ARCHIVE.exists():
    print(f"Restoring {ARCHIVE.stat().st_size / 1e9:.1f} GB from Drive ...")
    !tar -xf "{ARCHIVE}" -C /content
else:
    print("No archive on Drive yet: downloading every source (15-25 minutes).")
    !python -m crop_doctor.data.download --all
    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    !tar -cf "{ARCHIVE}" -C /content datasets
!python -m crop_doctor.data.download --check

## 3. Train once (~25–35 minutes)

Each epoch prints its validation macro-F1. The best epoch is chosen on validation photos, never on the test photos.

In [ ]:
!python -m crop_doctor.model.train_leaves

## 4. Save the model and results to Drive (so nothing needs retraining)

In [ ]:
import shutil, glob
OUT = Path("/content/drive/MyDrive/crop_doctor/exp12")
OUT.mkdir(parents=True, exist_ok=True)
for f in glob.glob("crop_doctor/outputs/exp12_leaf_crops*"):
    shutil.copy(f, OUT)
print(sorted(os.listdir(OUT)))

## 5. Results: before and after, on the same 476 PlantDoc test photos

In [ ]:
import json, pandas as pd
import matplotlib.pyplot as plt
r = json.loads(Path("crop_doctor/outputs/exp12_leaf_crops_results.json").read_text())
rows = [r["shipped_v3"], r["leaf_crops"], r["leaf_crops_plus_mirror"]]
table = pd.DataFrame([{
    "model": x["model"],
    "PlantDoc accuracy": f"{x['plantdoc_accuracy']:.1%}",
    "95% range": f"{x['plantdoc_95pct_range'][0]:.1%} - {x['plantdoc_95pct_range'][1]:.1%}",
    "PlantDoc macro-F1": round(x["plantdoc_macro_f1"], 3),
    "Tanzania accuracy": f"{x['tanzania_accuracy']:.1%}",
    "Overall accuracy": f"{x['all_accuracy']:.1%}",
} for x in rows])
display(table)
for name in ("gain_from_leaf_crops", "gain_from_both"):
    g = r[name]
    verdict = "clearly better" if g["clearly_better"] else "not clearly different from chance"
    print(f"{name}: {g['points']:+.1f} points (95% range {g['95pct_range_points'][0]:+.1f} to {g['95pct_range_points'][1]:+.1f}): {verdict}")

In [ ]:
per = pd.DataFrame({x["model"]: x["plantdoc_per_disease"] for x in (r["shipped_v3"], r["leaf_crops_plus_mirror"])})
per = per.sort_values(per.columns[0])
ax = per.plot.barh(figsize=(8, 5), width=0.8)
ax.set_xlabel("accuracy on PlantDoc test photos"); ax.set_xlim(0, 1)
ax.set_title("Per disease: shipped v3 vs leaf cut-outs + mirror averaging")
plt.tight_layout(); plt.show()

## 6. Reading the result

- **"Clearly better"** means the paired 95% range of the gain is above zero: the gain is unlikely to be luck.
  With 476 test photos, gains under about 4 points usually are not clear.
- Whatever the outcome, it is a valid finding for the report: either leaf-level training helps on field photos,
  or, at this data size, it does not.
- This run trains the model only. If it is adopted, the confidence settings (temperature and shortlist thresholds)
  must be refitted on its saved calibration predictions before it goes into the app.